# Learn and assign on dense data: human pancreas (Segerstolpe et al. 2016)

The same steps as the sparse notebook, on plate-based full-length data. What changes is the **technology preset** SCARLET chooses, which it prints in the pre-flight summary; you do not set it.

The steps:

1. the data, and the **pre-flight** summary SCARLET prints before it runs;
2. `learn` and `assign` in Python (each with its command-line twin);
3. the **report** of the annotated cells, with the held-out labels as labels supplied by you: the calls against
   them, the refusals and their reasons;
4. the **learn report**: the model's cross-validated self-assessment;
5. exporting the calls.

Runs in about two minutes on a laptop with 2 CPUs; the data are downloaded once (38 MB).

**Data:** Segerstolpe A, et al. *Cell Metabolism* 24, 593–607 (2016), ArrayExpress E-MTAB-5061: human pancreas, Smart-seq2 (about 6,400 detected genes per cell), as distributed by Abdelaal et al., *Genome Biology* 20, 194 (2019), Zenodo 3357167 (CC BY 4.0). The labels include the authors' own category 'co-expression'. These are the inputs of SCARLET's formal benchmark, task `A/pancreas_segerstolpe` (the benchmark's
class filter applied), compressed; their content is checked to be the benchmark's (`MANIFEST.tsv` and
`ATTRIBUTION.md` in the release assets). Guides: [learn](../docs/guide/learn.md), [assign](../docs/guide/assign.md),
[reports](../docs/reports/index.md), [reading the output](../docs/output.md).

## Setup

`scarlet` is the only package this notebook needs for the analysis itself. `NCPU` sets how many CPU cores SCARLET
may use (it never changes a result).

In [ ]:
import os

import scarlet

print("SCARLET", scarlet.__version__)
NCPU = min(4, os.cpu_count() or 1)
os.makedirs("out/segerstolpe", exist_ok=True)

## 1. The data

`scarlet.datasets.fetch` downloads the file once and checks its SHA-256 (a file already present with the right
checksum is not downloaded again).

In [ ]:
from scarlet import datasets

paths = datasets.fetch(["segerstolpe.h5ad"], dest="data")

SCARLET reads `.h5ad` files itself when given a path, with less memory than a plain read (integer counts held in
32 bits); `anndata` is used here only to look at the data and to make the split below. The data are raw counts (not
normalised, not log-transformed) with one label per cell in `obs['cell_type']`.

In [ ]:
import anndata as ad   # only to look at the data and split it

A = ad.read_h5ad(paths["segerstolpe.h5ad"])
print(A)
display(A.obs["cell_type"].value_counts().to_frame("cells"))

### A query with hidden labels

To show how SCARLET annotates cells it has never seen, and to score the result, we hold out one fifth of the
cells as a *query*: one fifth, as in one fold of the benchmark's five-fold cross-validation; **stratified** by
label, so every cell type keeps its share on both sides (small types would otherwise end up on one side only). The
query is written **without** its labels, so SCARLET cannot see them; we keep them aside as the truth to score
against. In normal use you have no labels for your query and skip this step: you learn on your labelled reference
and assign your own data. (The benchmark reports all five folds pooled, so its numbers differ from this one split.)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split   # only for the held-out split

ref_idx, qry_idx = train_test_split(np.arange(A.n_obs), test_size=0.2, stratify=A.obs["cell_type"], random_state=0)
reference, query = A[ref_idx].copy(), A[qry_idx].copy()
truth = query.obs["cell_type"].astype(str).copy()        # kept aside, for scoring only
query.obs = pd.DataFrame(index=query.obs_names)          # the query SCARLET sees has no labels
reference.write_h5ad("out/segerstolpe/reference.h5ad")
query.write_h5ad("out/segerstolpe/query.h5ad")
print(f"reference {reference.n_obs:,} cells, query {query.n_obs:,} cells")

## 2. Learn

`learn` finds each cell type's markers and writes one model file; every setting is at the package default. The
**pre-flight** summary at the top of its output says what SCARLET resolved before running: the number of cells,
genes and labels, the mean detected genes per cell, the **technology preset** (here *dense*, because the reference detects at least 5,000 genes per cell on average; the rule is printed with the number; the dense preset changes one setting of the marker search, the background penalty, because on deep data most genes are detected in a large share of all cells), the marker sizes and the blocklist (gene groups
such as ribosomal or mitochondrial genes kept out of the markers). Then one progress bar, and one line when the
model is written.

Command-line twin:
```
scarlet learn --reference out/segerstolpe/reference.h5ad --labels cell_type --model out/segerstolpe/segerstolpe.scarlet
```

In [ ]:
model = scarlet.learn("out/segerstolpe/reference.h5ad", "cell_type", "out/segerstolpe/segerstolpe.scarlet", ncpu=NCPU)

## 3. Assign

`assign` scores every query cell against every cell type of the model and against the background, and returns one
row per cell (also written into the annotated `.h5ad`):

- `scarlet_best`: the most likely cell type, or `unassigned` when the background explains the cell as well (the
  cell matches no cell type of the reference);
- `scarlet_unique`: the best cell type only when it is clearly better than the runner-up (FDR-adjusted, `alpha`
  0.01), else `unassigned` (the cell is ambiguous between two cell types);
- the **flip test**: when a call rests on genes the cell type never showed in the reference, `assign` refuses the
  cell (`unassigned`, reason *flip test*; marked in `scarlet_flip`);
- `scarlet_pe_low` (low positive evidence): a label, not a refusal.

Command-line twin:
```
scarlet assign --model out/segerstolpe/segerstolpe.scarlet --data out/segerstolpe/query.h5ad --output out/segerstolpe/query_annotated.h5ad
```

In [ ]:
calls = scarlet.assign(model, "out/segerstolpe/query.h5ad", "out/segerstolpe/query_annotated.h5ad", ncpu=NCPU)
display(calls.head())

## 4. The report of the annotated cells

`report` writes one self-contained HTML page (open it in a browser) and a bundle of tables beside it. To compare the
calls with the held-out labels **in the report itself**, we now add the labels to the annotated file, in a column
named `label_held_out`, and tell the report its name (`query_labels`). These labels are **for scoring only**:
`assign` has already run without them, and the report shows them in a section marked as labels supplied by you.

Command-line twin:
```
scarlet report out/segerstolpe/query_annotated.h5ad --model out/segerstolpe/segerstolpe.scarlet --query-labels label_held_out
```

In [ ]:
annotated = ad.read_h5ad("out/segerstolpe/query_annotated.h5ad")
annotated.obs["label_held_out"] = truth.reindex(annotated.obs_names).to_numpy()   # for scoring only
annotated.write_h5ad("out/segerstolpe/query_annotated.h5ad")

html_assign = scarlet.report("out/segerstolpe/query_annotated.h5ad", model="out/segerstolpe/segerstolpe.scarlet",
                             query_labels="label_held_out", ncpu=NCPU)
print("assign report:", html_assign)

### Reading the scores

Two numbers are read **together**: the share of cells SCARLET refused (`unassigned`), and the median over cell types
of the F1 on the cells it called. A refusal is not counted as an error: SCARLET says "I cannot place this cell"
instead of guessing, so a low F1 would mean wrong calls, and a high share refused would mean many cells without a
call. The report computes both (its section *Against labels supplied by the user*); the bundle holds them as tables,
which we read here.

In [ ]:
bundle = os.path.join(os.path.dirname(html_assign), "bundle", "tables")
display(pd.read_csv(os.path.join(bundle, "known_summary.tsv"), sep="\t")
        [["output", "cells", "pct_refused", "median_f1_called"]].round(3))

Every refused cell has a **reason**, in the order SCARLET applies its steps: *background* (no cell type fits
better than the background), *ambiguity* (the best cell type is not clearly better than the runner-up; unique call
only), *flip test* (the call rests on genes the cell type never showed). The report's per-cell table
(`cells.tsv`) holds the reason of each output; here, how many cells each reason covers:

In [ ]:
cells = pd.read_csv(os.path.join(bundle, "cells.tsv"), sep="\t")
for output in ("best", "unique"):
    reasons = cells[f"reason_{output}"].fillna("(called)")
    display(reasons.value_counts().to_frame(f"{output} call"))

## 5. The learn report: how well the model knows its own reference

The learn report estimates how well the model calls its own cell types by **cross-validation**: it learns on four
fifths of the reference and assigns the fifth left out, five times (once per fold; SCARLET prints one line per fold).
This is why a learn report takes longer than the others: five learns and five assigns, plus the UMAPs (here about a
minute). The weak-class table ranks the cell types weakest first by cross-validated recall; we show a few of its
columns (the report and the bundle have all of them).

Command-line twin:
```
scarlet report out/segerstolpe/segerstolpe.scarlet --reference out/segerstolpe/reference.h5ad
```

In [ ]:
html_learn = scarlet.report("out/segerstolpe/segerstolpe.scarlet", reference="out/segerstolpe/reference.h5ad", ncpu=NCPU)
print("learn report:", html_learn)
weak = pd.read_csv(os.path.join(os.path.dirname(html_learn), "bundle", "tables", "weak_classes.tsv"), sep="\t")
display(weak[["rank", "identity", "cells", "cv_recall_best", "cv_recall_unique", "most_confused_with"]].head(5)
        .round(3))

Several cell types have few cells here (MHC class II, for example): their markers come from few cells, and a held-out fold holds one or two of them, so their scores say little.

## 6. Export

The annotated query (`out/segerstolpe/query_annotated.h5ad`) keeps the input's counts and holds the calls in `obs`. For a plain
table of the calls, write the frame `assign` returned:

In [ ]:
calls.to_csv("out/segerstolpe/calls.tsv", sep="\t")
print("written:", os.path.abspath("out/segerstolpe/calls.tsv"))
display(pd.read_csv("out/segerstolpe/calls.tsv", sep="\t", index_col=0).head())